# Model fitting notebook
---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import scipy.optimize
from scipy.optimize import minimize
from scipy.stats import norm
from scipy.special import logsumexp
import warnings
from scipy.optimize import curve_fit
from scipy.optimize import approx_fprime
import scipy.stats as stats
from pybads import BADS

from matplotlib.colors import LinearSegmentedColormap

# TEMP
warnings.simplefilter("ignore", category=FutureWarning)

from src import *

# Set some defaults
plt.rc("axes.spines", top=False, right=False)
plt.rcParams["mathtext.default"] = "regular"
sns.set_theme(font_scale=1.2)
sns.set_style("ticks")
sns.set_palette('deep')

%config InlineBackend.figure_format = "retina"

---
## Load in data

In [ ]:
# Read in data to be fit
df = pd.read_csv("combined-data.csv")
df.head(10)

---
## Helper functions

---
## Fit data using MLE

In [ ]:
fit_single_subj = False

numtrials_adapt = 1540
numtrials_sdt = 800
numtrials_both = numtrials_adapt + numtrials_sdt

# We switch off the printing from BADS
options = {"display" : 'off'}

fit_task = "sdt"
n_sims = 1000
n_inits = 5

if fit_single_subj:
    # Pick your subject here
    subjects = np.array([1])
else:
    subjects = df['SN'].unique()


In [ ]:
%%time
# PIECE parameter bounds: alpha, beta, sigma_p, sigma_pert, b,
bounds       = ((0.1, 5), (0.001, 1), (0.1, 10), (0.1, 120), (-5, 5))
plaus_bounds = ((0.5, 4), (0.01, 0.5), (0.3, 6), (3, 20), (-2.5, 2.5))

results_list = []
aic_list = []
bic_list = []
for s in subjects:
    fits_piece, aic, bic = fit_bads("piece", s, df, bounds, plaus_bounds, n_inits=n_inits, n_sims=n_sims, fit_task=fit_task, options=options)
    results_list.append(fits_piece)
    aic_list.append(aic)
    bic_list.append(bic)

best_params_piece = np.array([res.x for res in results_list]).squeeze()

fits_piece = pd.DataFrame({
    'SN': subjects,
    'model': "piece",
    "fit_task": fit_task,
    'nll': [res.fval for res in results_list],
    "aic": aic_list,
    "bic": bic_list,
    'params': [res.x for res in results_list]
})

# Convert the list of dictionaries into a DataFrame
fits_piece[["alpha", "beta", "sigma_p", "sigma_pert", "b"]] = pd.DataFrame(fits_piece['params'].tolist(), index=fits_piece.index)

if fit_task == "adapt":
    fits_piece_adapt = fits_piece.copy()
elif fit_task == "both":
    fits_piece_both = fits_piece.copy()
elif fit_task == "sdt": 
    fits_piece_sdt = fits_piece.copy()

# # Checks path and creates new csv
# save_csv(fits_piece, model='piece')
# fits_piece.to_csv("model-fits/nolapse_piece_simult.csv", index=False)
# fits_piece.to_csv("model-fits/nolapse_piece_adapt.csv", index=False)


In [ ]:
%%time
# Fit VCO
if fit_task == "sdt":
    # Parameter bounds (C, sigma_v, b) 
    bounds       = [(0.1, 10), (0.1, 10.0), (-5, 5)]
    plaus_bounds = [(0.5, 5), (0.5, 6), (-2.5, 2.5)]
else:
    # Parameter bounds (C, B, sigma_v, b) 
    bounds       = [(0.1, 10), (0.01, 1), (0.1, 10.0), (-5, 5)]
    plaus_bounds = [(0.5, 5), (0.05, 0.6), (0.5, 6), (-2.5, 2.5)]

results_list = []
aic_list = []
bic_list = []
for s in subjects:
    fits_vco, aic, bic = fit_bads("vco", s, df, bounds, plaus_bounds, n_inits=n_inits, n_sims=n_sims, 
                                  fit_task=fit_task, options=options)
    results_list.append(fits_vco)
    aic_list.append(aic)
    bic_list.append(bic)

best_params_vco = np.array([res.x for res in results_list]).squeeze()

fits_vco = pd.DataFrame({
    'SN': subjects,
    'model': "vco",
    "fit_task": fit_task,
    'nll': [res.fval for res in results_list],
    "aic": aic_list,
    "bic": bic_list,
    'params': [res.x for res in results_list]
})

# Convert the list of dictionaries into a DataFrame
if fit_task == "adapt":
    fits_vco[["C", "B", "sigma_v", "b"]] = pd.DataFrame(fits_vco['params'].tolist(), index=fits_vco.index)
    fits_vco_adapt = fits_vco.copy()
elif fit_task == "both":
    fits_vco[["C", "B", "sigma_v", "b"]] = pd.DataFrame(fits_vco['params'].tolist(), index=fits_vco.index)
    fits_vco_both = fits_vco.copy()
elif fit_task == "sdt": 
    fits_vco[["C", "sigma_v", "b"]] = pd.DataFrame(fits_vco['params'].tolist(), index=fits_vco.index)
    fits_vco_sdt = fits_vco.copy()

# # Checks path and creates new csv
# save_csv(fits_vco, model='vco')
# fits_vco.to_csv("model-fits/fixed_vco_simult.csv", index=False)
# fits_vco.to_csv("model-fits/fixed_vco_adapt.csv", index=False)


In [ ]:
%%time
# Fit CO model
if fit_task == "sdt":
    # Parameter bounds (C, sigma_comb) 
    bounds = [(0.1, 10), (0.1, 10.0)]
    plaus_bounds = [(0.5, 5), (0.5, 6)]
else:
    # Parameter bounds (C, B, sigma_comb, b) 
    bounds       = [(0.1, 10), (0.01, 1), (0.1, 10.0), (-5, 5)]
    plaus_bounds = [(0.5, 5), (0.05, 0.6), (0.5, 6), (-2.5, 2.5)]

results_list = []
aic_list = []
bic_list = []
for s in subjects:
    fits_co, aic, bic = fit_bads("co", s, df, bounds, plaus_bounds, n_inits=n_inits, n_sims=n_sims, fit_task=fit_task, options=options)
    results_list.append(fits_co)
    aic_list.append(aic)
    bic_list.append(bic)

fits_co = pd.DataFrame({
    'SN': subjects,
    'model': "co",
    "fit_task" : fit_task,
    'nll': [res.fval for res in results_list],
    "aic": aic_list,
    "bic": bic_list,
    'params': [res.x for res in results_list]
})

# Convert the list of dictionaries into a DataFrame
if fit_task == "adapt":  
    fits_co[["C", "B", "sigma_comb", "b"]] = pd.DataFrame(fits_co['params'].tolist(), index=fits_co.index)
    fits_co_adapt = fits_co.copy()
elif fit_task == "both":
    fits_co[["C", "B", "sigma_comb", "b"]] = pd.DataFrame(fits_co['params'].tolist(), index=fits_co.index)
    fits_co_both = fits_co.copy()
elif fit_task == "sdt": 
    fits_co[["C", "sigma_comb"]] = pd.DataFrame(fits_co['params'].tolist(), index=fits_co.index)
    fits_co_sdt = fits_co.copy()

# # Checks path and creates new csv
# save_csv(fits_co, model='co')
# fits_co.to_csv("model-fits/nolapse_co_simult.csv", index=False)
# fits_co.to_csv("model-fits/nolapse_co_adapt.csv", index=False)


---
## Misc code

In [ ]:

# # Fit EGE-adaptation data with breakpoint regression
# x_fit, y_fit, ci_lower, ci_upper, popt, pcov = fit_piecewise_with_ci(df_vmr, df_vmr["SN"], xcol="rotation", ycol="adaptation", 
#                                                                         bp1=6, bp2=-6, n_points=300, alpha=0.05)


# # Plot data for adapt vs vmr-ege
# fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(8, 3.5))
# # sns.regplot(data=df_vmr, x="rotation", y="adaptation", x_jitter=0.1, 
# #            scatter_kws=dict(alpha=0.3, color='gray', s=20, ), 
# #            line_kws=dict(linewidth=2, color='red'), ci=95, ax=ax1)
# sns.stripplot(data=df_vmr, x="rotation", y="adaptation", color="gray", alpha=0.3, s=3, native_scale=True, ax=ax1)
# # sns.lineplot(data=df_vmr, x="rotation", y="adaptation", linewidth=2, err_style="bars", errorbar=("ci", 95), c="r", ax=ax1)
# ax1.plot(x_fit, y_fit, c="r", linewidth=3)


# ax1.axhline(0, c="k", linewidth=0.5)
# ax2.axhline(0, c="k", linewidth=0.5)
# ax1.set(xticks=np.arange(-10, 10.1, 2), yticks=np.arange(-10, 10.1, 2), xlabel="Rotation [EGE] ($\degree$)", 
#         xlim=(-12.5, 12.5), ylim=(-10, 10), ylabel="Adaptation ($\degree$)")
# sns.despine()



# def vco_ll(df, C, B, sigma_v, b, n_sims):
    
#     # Grab vars from dataframe (2 different sigma_motors)
#     sigma_f_adapt = df.loc[df["session"] == "adapt", "sigma_motor_adapt"].values[0]
#     sigma_f_sdt = df.loc[df["session"] == "sdt", "sigma_motor_sdt"].values[0]
    
#     rotation = df["delta_xv"].values
#     x_f = df["x_f"].values
#     vis_fb = df["vis_fb"].values
#     pert_resp = df["pert_resp"].values
#     session = df["session"].values

#     num_trials = len(x_f)
#     trialnum_start_adapt = 101
#     trialnum_end_adapt = 1640
    
#     no_fit_idx = df["no_fit"].values
    
#     # Initialize 
#     log_p = []
    
#     # Loop through trials
#     for i in range(num_trials):
#         if session[i] == "adapt":
#             sigma_f = sigma_f_adapt
#         elif session[i] == "sdt":
#             sigma_f = sigma_f_sdt
            
#         # MC simulation of observations
#         if vis_fb[i] == 0:
#             x_v = np.ones(n_sims) * b  # ensures zero err and no detection
#         else:
#             x_v = np.random.normal(loc=x_f[i] + rotation[i], scale=sigma_v, size=n_sims)
            
#         err = x_v - b
#         det_idx = np.abs(err) > C
#         p_det = (det_idx.sum()) / n_sims
#         p_det = np.clip(p_det, 1e-6, 1 - 1e-6)
#         p_nodet = 1 - p_det
        
#         pred_reach = np.random.normal(loc=b, scale=sigma_f, size=n_sims)
#         pred_reach[det_idx] = -B * err[det_idx] + pred_reach[det_idx]
            
#         # Fit a KDE to your simulated Z data array
#         kde = stats.gaussian_kde(pred_reach)
        
#         if session[i] == "sdt":
#             if pert_resp[i] == 1:
#                 log_p.append(np.log(p_det))
#             elif pert_resp[i] == 0:
#                 log_p.append(np.log(p_nodet))
        
#         elif session[i] == "adapt":
#             if (df.loc[i, "TN"] == trialnum_start_adapt):  # only append if sdt task was first
#                 log_p.append(0)  # bc there is no prediction for first adapt trial
            
#             if df.loc[i, "TN"] == trialnum_end_adapt:
#                 log_p = log_p  # don't make mistake of fitting first SDT trial (x_f[i + 1])
#             else:
#                 # Each appended log-prob is p(next trial data|current trial, params, model)
#                 log_p.append(np.log(kde.evaluate(x_f[i + 1])[0]))
        
#     log_p = np.array(log_p)
#     log_p[no_fit_idx] = 0
#     total_log_p = np.sum(log_p)

#     return total_log_p


# def fit_vco(subject_id, df, bounds, n_inits, n_sims, fit_task="both"):
    
#     subj_df = (df[df['SN'] == subject_id]).reset_index(drop=True)
#     if fit_task == "sdt":
#         subj_df = subj_df[subj_df["session"] == "sdt"]
#     elif fit_task == "adapt":
#         subj_df = subj_df[subj_df["session"] == "adapt"]        
    
#     best_res = None
#     for _ in range(n_inits):
#         x0 = np.array([np.random.uniform(low, high) for low, high in bounds])
#         res = scipy.optimize.minimize(
#             fun=nll_vco,
#             x0=x0,
#             bounds=bounds,
#             args=(subj_df, n_sims, fit_task),
#             method="Powell",
#             options={'disp': True}
#         )

#         if best_res is None or res.fun < best_res.fun:
#             best_res = res
            
#     return best_res


# def fit_co(subject_id, df, bounds, n_inits, n_sims):
    
#     subj_df = (df[df['SN'] == subject_id]).reset_index(drop=True)
    
#     best_res = None
#     for _ in range(n_inits):
#         x0 = np.array([np.random.uniform(low, high) for low, high in bounds])
#         res = scipy.optimize.minimize(
#             fun=nll_co,
#             x0=x0,
#             bounds=bounds,
#             args=(subj_df, n_sims),
#             method="Powell",
#             options={'disp': True}
#         )

#         if best_res is None or res.fun < best_res.fun:
#             best_res = res
            
#     return best_res


# fits_vco = Parallel(n_jobs=1)(
#     delayed(fit_subj)(nll_vco, s, df, bounds, n_inits=1, n_sims=10, fit_task=fit_task)
#     for s in subjects
# )

---
## Assessing Model Fits

In [ ]:
# # Change if you want to look at someone else's data
# test_subj = 14
# # Create data frame for testing best-fit params
# test_df = (df[df['SN'] == test_subj]).reset_index(drop=True)

# # Isolate the adaptation session
# if fit_task == "adapt":
#     test_df = test_df[test_df["session"] == "adapt"].reset_index(drop=True)
# elif fit_task == "sdt":
#     test_df = test_df[test_df["session"] == "sdt"].reset_index(drop=True)

# # Assess model fit
# total_ll_piece, pertrial_ll_piece = piece_ll(test_df, best_params_piece[0], best_params_piece[1], 
#                                                   best_params_piece[2], best_params_piece[3], best_params_piece[4],
#                                                   n_sims, fit_task)


# print(fits_piece["NLL"], total_ll_piece)
# print(np.median(pertrial_ll_piece), np.max(pertrial_ll_piece))
# fig, ax = plt.subplots()
# ax.hist(pertrial_ll_piece, bins=50)
# plt.show()

# print(f"Lowest LL: {np.min(pertrial_ll_piece):.3f}; Num: {(pertrial_ll_piece == np.min(pertrial_ll_piece)).sum():.3f}")

# # Assess model fit
# best_params_vco = np.array([res.x for res in results_list]).squeeze()
# best_params_vco = best_params_vco[13]
# total_ll_vco, pertrial_ll_vco = vco_ll(test_df, best_params_vco[0], best_params_vco[1], 
#                                                   best_params_vco[2], best_params_vco[3],
#                                                   n_sims, fit_task)

# print(fits_vco["NLL"], total_ll_vco)

# # Assess model fit
# best_params_co = np.array([res.x for res in results_list]).squeeze()
# total_ll_co, pertrial_ll_co = co_ll(test_df, best_params_co[0], best_params_co[1], 
#                                                   best_params_co[2], best_params_co[3],
#                                                   n_sims, fit_task)

# print(fits_co["NLL"], total_ll_co)

# print(np.median(pertrial_ll_co), np.max(pertrial_ll_co))
# fig, ax = plt.subplots()
# ax.hist(pertrial_ll_co, bins=50)
# plt.show()

# print(f"Lowest LL: {np.min(pertrial_ll_co):.3f}; Num: {(pertrial_ll_co == np.min(pertrial_ll_co)).sum()}")